In [ ]:
import win32com.client as win32

In [2]:
aspen = win32.Dispatch("Apwn.Document")

In [3]:
aspen.InitFromArchive2(r"C:\Users\ravik\Desktop\Digital Twin\CSTR 2\CSTR simulation 2.bkp")
aspen.Visible = 1

In [4]:
CSTR = aspen.Tree.Findnode(r"\Data\Blocks\CSTR")

In [5]:
# CSTR_input = aspen.Tree.Findnode(r"\Data\Blocks\CSTR\Input")

# for child in CSTR_input.Elements:
#     print(child.Name)

In [6]:
reactor_temp =  aspen.Tree.Findnode(r"\Data\Blocks\CSTR\Input\TEMP")

In [7]:
cstr_pressure =  aspen.Tree.Findnode(r"\Data\Blocks\CSTR\Input\PRES")

In [8]:
reactor_vol =  aspen.Tree.Findnode(r"\Data\Blocks\CSTR\Input\VOL")

In [9]:
Feed1_temp= aspen.Tree.FindNode(r"\Data\Streams\FEED1\Input\TEMP\MIXED")

In [10]:
H2_flow_in = aspen.Tree.FindNode(r"\Data\Streams\FEED2\Input\TOTFLOW\MIXED")

In [11]:
glycerol_frac_feed1= aspen.Tree.FindNode(r"\Data\Streams\FEED1\Input\FLOW\MIXED\GLYCEROL")
water_frac_feed1 = aspen.Tree.FindNode(r"\Data\Streams\FEED1\Input\FLOW\MIXED\WATER")
# glycerol_frac_feed1.value= 0.3
# water_frac_feed1.value = 1.0-glycerol_frac_feed1.value


In [12]:
feed1_flowrate = aspen.Tree.FindNode(r"\Data\Streams\FEED1\Input\TOTFLOW\MIXED")
print(feed1_flowrate.value)

174.46276288


In [13]:
gly_frac_product = aspen.Tree.FindNode("\Data\Streams\PRODUCT\Output\MOLEFRAC\MIXED\GLYCEROL")
gly_frac_product.value

0.435805882

In [14]:
product_flowrate = aspen.Tree.FindNode("\Data\Streams\PRODUCT\Output\MOLEFLMX\MIXED")
product_flowrate.value

194.824634

In [15]:
PDO_frac = aspen.Tree.FindNode("\Data\Streams\PRODUCT\Output\MOLEFRAC\MIXED\PDO")
PDO_frac.value

0.103989961

In [16]:
gly_conv = (((feed1_flowrate.value*glycerol_frac_feed1.value)- (product_flowrate.value*gly_frac_product.value))/(feed1_flowrate.value*glycerol_frac_feed1.value))*100
gly_conv

19.264683368280437

In [17]:
import numpy as np
import pandas as pd
import scipy as sp
from scipy.stats import qmc

In [19]:
# df = pd.DataFrame(samples_scaled ,columns=["R_temp","R_vol","H2_flow","feed1_flow","frac_gly_in"])
# df.shape

# df.to_excel(r"C:\Users\ravik\Desktop\Digital Twin\CSTR2_data\cstr_features.xlsx", index= False)

In [20]:
df  = pd.read_excel("cstr_features.xlsx")

In [21]:
df.shape

(1000000, 5)

In [22]:
aspen.Engine.Run2()

In [ ]:
# Cross verification of the collected data
#  H2_flow_in.value = 23.183506
# reactor_temp.value = 294.149009
# reactor_vol.value = 1802.541737
# feed1_flowrate.value = 120.007883
# glycerol_frac_feed1.value =0.128399
# water_frac_feed1.value = 1 - 0.174281

# aspen.Engine.Run2()
# print(PDO_frac.value)

In [24]:
import time

gly_conv_results = []
PDO_fraction = []

MAX_TIME = 60   # seconds

for i in range(4000, 5000):

    try:

        # Set inputs
        H2_flow_in.value = df["H2_flow"][i]
        reactor_temp.value = df["R_temp"][i]
        reactor_vol.value = df["R_vol"][i]
        feed1_flowrate.value = df["feed1_flow"][i]

        glycerol_frac_feed1.value = df["frac_gly_in"][i]
        water_frac_feed1.value = 1 - glycerol_frac_feed1.value

        # Start timer
        start_time = time.time()

        # Run Aspen
        aspen.Engine.Run2()

        # Wait for Aspen to stabilize
        time.sleep(0.5)

        # Check elapsed time
        elapsed_time = time.time() - start_time

        if elapsed_time > MAX_TIME:
            print(f"Stopping: Aspen took {elapsed_time:.2f} seconds at row {i}")
            break

        # Read outputs
        gly_conv = (
            (
                feed1_flowrate.value * glycerol_frac_feed1.value
                - product_flowrate.value * gly_frac_product.value
            )
            / (feed1_flowrate.value * glycerol_frac_feed1.value)
        ) * 100

        gly_conv_results.append(gly_conv)
        PDO_fraction.append(PDO_frac.value)

        print(f"Completed row {i} in {elapsed_time:.2f} seconds")

    except Exception as e:

        print(f"Failed at row {i}")
        print(e)
        break

Completed row 4000 in 3.00 seconds
Completed row 4001 in 2.00 seconds
Completed row 4002 in 5.12 seconds
Completed row 4003 in 3.09 seconds
Completed row 4004 in 2.06 seconds
Completed row 4005 in 2.00 seconds
Completed row 4006 in 1.96 seconds
Completed row 4007 in 1.89 seconds
Completed row 4008 in 2.09 seconds
Completed row 4009 in 1.78 seconds
Completed row 4010 in 1.95 seconds
Completed row 4011 in 1.85 seconds
Completed row 4012 in 1.84 seconds
Completed row 4013 in 1.98 seconds
Completed row 4014 in 1.91 seconds
Completed row 4015 in 2.00 seconds
Completed row 4016 in 1.89 seconds
Completed row 4017 in 1.88 seconds
Completed row 4018 in 1.85 seconds
Completed row 4019 in 1.83 seconds
Completed row 4020 in 1.83 seconds
Completed row 4021 in 1.80 seconds
Completed row 4022 in 1.88 seconds
Completed row 4023 in 1.76 seconds
Completed row 4024 in 1.76 seconds
Completed row 4025 in 1.78 seconds
Completed row 4026 in 4.32 seconds
Completed row 4027 in 1.73 seconds
Completed row 4028 i

In [25]:
PDO_fraction_arr = np.array(PDO_fraction)
PDO_fraction_arr.shape

(1000,)

In [26]:
gly_conv_results_arr = np.array(gly_conv_results)
gly_conv_results_arr.shape

(1000,)

In [27]:
gly_conv_results_arr = gly_conv_results_arr.reshape(-1,1)

In [28]:
PDO_fraction_arr = PDO_fraction_arr.reshape(-1,1)

In [30]:
df_10= pd.DataFrame(df.iloc[4000:5000,:])
df_10.shape

(1000, 5)

In [31]:
df_10["PDO_fraction"] = PDO_fraction_arr
df_10["gly_conv"] = gly_conv_results_arr

In [32]:
df_10.to_excel(r"C:\Users\ravik\Desktop\Digital Twin\CSTR2_data\cstr_data_4000_to_5000.xlsx", index= False)

In [61]:
# df_1 = pd.DataFrame(df.iloc[0:100,:])

In [62]:
# df_1["PDO_fraction"] = PDO_fraction_arr
# df_1["gly_conv"] = gly_conv_results_arr

In [65]:
# df_1.to_excel(r"C:\Users\ravik\Desktop\Digital Twin\CSTR2_data\cstr_data_0_to_100.xlsx", index= False)

In [ ]:
# feed1_flowrate.value,reactor_vol.value, reactor_temp.value,H2_flow_in.value = 105.819781,1221.163810,236.340806,14.044374
# aspen.Engine.Run2()
# gly_conv